In [ ]:
pip install torch plotly yfinance statsmodels scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=b13e0351a645450407421428574495dd0bbf094f8c62b49fd1f192e3cb7505f9
  Stored in directory: /root/.cache/pip/wheels/a1/d7/29/7781cc5eb9a3659d032d7d15bdd0f49d07d2b24fec29f44bc4
Successfully built ta


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import DataLoader, TensorDataset, random_split
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import scipy.stats as stats

import warnings
import traceback
import os


warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", message="The verbose parameter is deprecated")
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning, module='statsmodels')


SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class QuantumInspiredLayer(nn.Module):
    def __init__(self, input_size, output_size, theta_phi_init_scale=0.05, noise_std=0.01):
        super(QuantumInspiredLayer, self).__init__()
        self.noise_std = noise_std
        self.theta = nn.Parameter(torch.randn(input_size, output_size, dtype=torch.float32) * theta_phi_init_scale)
        self.phi = nn.Parameter(torch.randn(input_size, output_size, dtype=torch.float32) * theta_phi_init_scale)

    def forward(self, x):

        x_float = x.to(torch.float32)


        if self.noise_std > 0:
            theta_noise = torch.randn_like(self.theta) * self.noise_std
            phi_noise = torch.randn_like(self.phi) * self.noise_std
            current_theta = self.theta + theta_noise
            current_phi = self.phi + phi_noise
        else:
            current_theta = self.theta
            current_phi = self.phi


        psi_real = torch.cos(current_theta)
        psi_imag = torch.sin(current_phi)
        psi = torch.complex(psi_real, psi_imag) # psi is now explicily torch.complex64


        x_expanded = x_float.unsqueeze(-1)
        psi_expanded = psi.unsqueeze(0).unsqueeze(0)    # Accounts for batch and sequence dimensions if present

        output_product_complex = x_expanded * psi_expanded

        output_sum_complex = torch.sum(output_product_complex, dim=2) # Sum over the original input_size dimension

        return output_sum_complex

class EnhancedQuantumInspiredLSTM(nn.Module):
    def __init__(self, input_size, hidden_size_lstm, num_layers_lstm, output_size_final, dropout_rate=0.2,
                 quantum_complex_output_dim=32,
                 theta_phi_init_scale=0.05, q_noise_std=0.01):
        super(EnhancedQuantumInspiredLSTM, self).__init__()
        self.hidden_size_lstm = hidden_size_lstm
        self.num_layers_lstm = num_layers_lstm

        self.quantum_layer = QuantumInspiredLayer(input_size, quantum_complex_output_dim,
                                                  theta_phi_init_scale=theta_phi_init_scale,
                                                  noise_std=q_noise_std)

        lstm_input_size = quantum_complex_output_dim * 2

        self.lstm = nn.LSTM(lstm_input_size, hidden_size_lstm, num_layers_lstm,
                            batch_first=True, dropout=dropout_rate if num_layers_lstm > 1 else 0)

        self.fc1 = nn.Linear(hidden_size_lstm, hidden_size_lstm // 2)
        self.act1 = nn.GELU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size_lstm // 2, output_size_final)

    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        # x_q_complex shape after quantum_layer: (batch_size, seq_length, quantum_complex_output_dim)
        x_q_complex = self.quantum_layer(x)

        # x_for_lstm shape: (batch_size, seq_length, quantum_complex_output_dim * 2)
        x_for_lstm = torch.cat((x_q_complex.real, x_q_complex.imag), dim=-1)

        lstm_out, _ = self.lstm(x_for_lstm.to(torch.float32)) # LSTM expects float32 input
        out = lstm_out[:, -1, :] # Take output of last time step
        out = self.fc1(out); out = self.act1(out); out = self.dropout1(out)
        out = self.fc2(out)
        return out

class StandardLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_rate=0.2):
        super(StandardLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.act1 = nn.GELU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size // 2, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        lstm_out, _ = self.lstm(x.to(torch.float32)) # lstm_out shape: (batch_size, seq_length, hidden_size)

        # Take the output from the last time step
        out = lstm_out[:, -1, :] # out shape: (batch_size, hidden_size)

        out = self.fc1(out); out = self.act1(out); out = self.dropout1(out)
        out = self.fc2(out)
        return out


def fetch_data_and_vix(symbols, vix_ticker, start_date, end_date):
    """Fetches stock Adj Close and VIX Close, returns aligned DataFrame."""
    print(f"--- Fetching Data ---")
    all_symbols_to_fetch = symbols + [vix_ticker]
    print(f"Assets: {', '.join(all_symbols_to_fetch)}")
    print(f"Period: {start_date:%Y-%m-%d} to {end_date:%Y-%m-%d}")
    try:
        data_full = yf.download(all_symbols_to_fetch, start=start_date, end=end_date, progress=True, timeout=90, auto_adjust=False)
        if data_full.empty:
            raise ValueError("Data download returned an empty DataFrame.")

        extracted_data = {}
        for sym in symbols:
            adj_col = ('Adj Close', sym)
            close_col = ('Close', sym)
            if adj_col in data_full.columns and not data_full[adj_col].isnull().all():
                extracted_data[sym] = data_full[adj_col]
            elif close_col in data_full.columns and not data_full[close_col].isnull().all():
                print(f"    Warn: Using 'Close' instead of 'Adj Close' for {sym}.")
                extracted_data[sym] = data_full[close_col]
            else:
                print(f"    Warn: Skipping {sym} due to missing 'Adj Close'/'Close' data.")

        vix_adj_col = ('Adj Close', vix_ticker)
        vix_close_col = ('Close', vix_ticker)
        if vix_close_col in data_full.columns and not data_full[vix_close_col].isnull().all():
            extracted_data[vix_ticker] = data_full[vix_close_col]
        elif vix_adj_col in data_full.columns and not data_full[vix_adj_col].isnull().all():
            print(f"    Warn: Using 'Adj Close' for VIX ({vix_ticker}) as 'Close' is unavailable/all NaNs.")
            extracted_data[vix_ticker] = data_full[vix_adj_col]
        elif vix_ticker in data_full.columns and data_full.columns.nlevels == 1: # VIX downloaded as single series
             if 'Close' in data_full.columns and not data_full['Close'].isnull().all():
                 extracted_data[vix_ticker] = data_full['Close']
             elif 'Adj Close' in data_full.columns and not data_full['Adj Close'].isnull().all():
                 extracted_data[vix_ticker] = data_full['Adj Close']
             else:
                raise ValueError(f"Could not extract valid 'Close' or 'Adj Close' for VIX ({vix_ticker}) (single series).")
        else:
            raise ValueError(f"Could not find valid 'Close' or 'Adj Close' column for VIX ({vix_ticker}) in the downloaded data.")

        if not extracted_data or len(extracted_data) < len(all_symbols_to_fetch):
            missing_syms = set(all_symbols_to_fetch) - set(extracted_data.keys())
            raise ValueError(f"Could not extract data for all requested symbols. Missing: {missing_syms}")

        combined_df = pd.DataFrame(extracted_data)
        initial_rows = len(combined_df)
        data_cleaned = combined_df.dropna()
        dropped_rows = initial_rows - len(data_cleaned)
        if dropped_rows > 0: print(f"Dropped {dropped_rows} rows due to missing values after combining.")
        if data_cleaned.empty: raise ValueError("Data is empty after removing NaNs.")
        print(f"--- Data Fetching Complete --- Final Shape: {data_cleaned.shape}")
        return data_cleaned
    except Exception as e:
        error_message = f"Data download or processing failed: {e}"
        print(f"Full data columns: {data_full.columns if 'data_full' in locals() else 'Not available'}")
        if isinstance(e, KeyError): error_message = f"Data processing failed - KeyError accessing column: {e}."
        raise ConnectionError(error_message) from e

def prepare_train_val_data(data, feature_cols, target_cols, train_end_date, seq_length, val_split_ratio=0.15, batch_size=64):
    print(f"--- Preparing Train/Val Data (Split Date: {train_end_date:%Y-%m-%d}) ---")
    data = data.sort_index()
    train_val_data = data[data.index <= train_end_date].copy()
    if train_val_data.empty: raise ValueError("No training/validation data found before the split date.")
    if len(train_val_data) < seq_length + 2: raise ValueError(f"Training/validation data too short ({len(train_val_data)} rows) for seq_length {seq_length}.")

    feature_scaler = StandardScaler()
    target_scaler = MinMaxScaler(feature_range=(0, 1))
    feature_scaler.fit(train_val_data[feature_cols])
    target_scaler.fit(train_val_data[target_cols])

    scaled_tv_features = feature_scaler.transform(train_val_data[feature_cols])
    scaled_tv_targets = target_scaler.transform(train_val_data[target_cols])
    x_tv, y_tv = [], []
    for i in range(len(scaled_tv_features) - seq_length):
        x_tv.append(scaled_tv_features[i : i + seq_length])
        y_tv.append(scaled_tv_targets[i + seq_length])
    if not x_tv: raise ValueError("Failed to create sequences for training/validation.")

    x_tv_tensor = torch.FloatTensor(np.array(x_tv)).to(device)
    y_tv_tensor = torch.FloatTensor(np.array(y_tv)).to(device)
    dataset = TensorDataset(x_tv_tensor, y_tv_tensor)
    total_size = len(dataset)
    val_size = int(total_size * val_split_ratio)
    train_size = total_size - val_size
    if train_size <= 0 or val_size <= 0:
        raise ValueError(f"Cannot split dataset. Train size: {train_size}, Val size: {val_size}")

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False)
    print("DataLoaders created.")
    return train_loader, val_loader, feature_scaler, target_scaler


def train_model_with_val(model, model_name, train_loader, val_loader, learning_rate, weight_decay, num_epochs_max, patience_early_stopping=25):
    """Trains model with validation loop, early stopping, and returns loss history."""
    print(f"\n--- Starting Training for {model_name} ---")
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5, verbose=False)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    train_losses, val_losses = [], []
    model_save_path = f'{model_name}_best_checkpoint.pth'

    for epoch in range(num_epochs_max):
        model.train()
        total_train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader) if len(train_loader) > 0 else 0.0
        train_losses.append(avg_train_loss)

        model.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                total_val_loss += loss.item()
        avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else float('inf')
        val_losses.append(avg_val_loss)
        scheduler.step(avg_val_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f'{model_name} - Epoch [{epoch+1}/{num_epochs_max}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}, LR: {current_lr:.4g}')

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            try: torch.save(model.state_dict(), model_save_path)
            except Exception as e: print(f"Warn: Failed to save checkpoint {model_save_path}: {e}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience_early_stopping:
                print(f'{model_name} Early stopping triggered at Epoch {epoch+1}. Best Val Loss: {best_val_loss:.6f}')
                break
    print(f"--- Training Finished for {model_name} ---")
    try:
        if os.path.exists(model_save_path):
            model.load_state_dict(torch.load(model_save_path, map_location=device))
            print(f"Loaded best model checkpoint ({model_save_path}) with Val Loss: {best_val_loss:.6f}")
        else: print(f"Warn: No checkpoint found at {model_save_path}. Using model from last epoch.")
    except Exception as e: print(f"Warn: Error loading checkpoint {model_save_path}: {e}. Using model from last epoch.")
    return model, train_losses, val_losses


def evaluate_period(model, model_name, data, feature_cols, target_cols, eval_start_date, eval_end_date, seq_length, feature_scaler, target_scaler):
    """ Evaluates model performance historically, predicting one day ahead. Returns overall daily errors."""
    print(f"\n--- Evaluating {model_name} Period: {eval_start_date:%Y-%m-%d} to {eval_end_date:%Y-%m-%d} ---")
    model.eval().to(device)
    data = data.sort_index()
    try:
        first_prediction_day_loc = data.index.searchsorted(eval_start_date)
        input_data_start_loc = first_prediction_day_loc - seq_length
        actual_data_end_loc = data.index.searchsorted(eval_end_date, side='right')

        if input_data_start_loc < 0:
            new_eval_start_idx = seq_length
            if new_eval_start_idx >= actual_data_end_loc :
                 raise ValueError("Evaluation period too short after adjusting for history, or insufficient data overall.")
            eval_start_date = data.index[new_eval_start_idx]
            first_prediction_day_loc = new_eval_start_idx
            input_data_start_loc = 0
            print(f"Adjusted eval start date to {eval_start_date:%Y-%m-%d}")

        eval_subset = data.iloc[input_data_start_loc : actual_data_end_loc]
        prediction_dates = data.index[first_prediction_day_loc : actual_data_end_loc]
        if prediction_dates.empty: raise ValueError(f"No data found for prediction in the evaluation period {eval_start_date} to {eval_end_date}.")
        if eval_subset.empty: raise ValueError(f"Evaluation subset is empty. Check date ranges and data availability.")
    except Exception as e:
         print(f"Error setting up evaluation period for {model_name}: {e}")
         return pd.DataFrame(), pd.DataFrame()

    scaled_features_subset = feature_scaler.transform(eval_subset[feature_cols])
    actual_prices_for_period = eval_subset[target_cols].iloc[seq_length:].values # This is a NumPy array
    predictions_list, actuals_list, dates_list = [], [], []

    with torch.no_grad():
        for i in range(len(prediction_dates)):
            input_sequence_scaled = scaled_features_subset[i : i + seq_length]
            if input_sequence_scaled.shape[0] < seq_length:
                print(f"Warn: Skipping prediction for {prediction_dates[i]} due to insufficient lookback data.")
                continue
            input_tensor = torch.FloatTensor(input_sequence_scaled).unsqueeze(0).to(device)
            predicted_scaled = model(input_tensor)
            predicted_price = target_scaler.inverse_transform(predicted_scaled.cpu().numpy())[0] # This is a 1D NumPy array
            actual_price = actual_prices_for_period[i] # This is also a 1D NumPy array
            predictions_list.append(predicted_price)
            actuals_list.append(actual_price)
            dates_list.append(prediction_dates[i])

    if not dates_list:
        print(f"Warn: No predictions generated for {model_name} in period {eval_start_date} to {eval_end_date}.")
        return pd.DataFrame(), pd.DataFrame()

    pred_df = pd.DataFrame(np.array(predictions_list), index=pd.DatetimeIndex(dates_list), columns=[f'{col}_Pred' for col in target_cols])
    actual_df = pd.DataFrame(np.array(actuals_list), index=pd.DatetimeIndex(dates_list), columns=[f'{col}_Actual' for col in target_cols])
    details_df = pd.concat([pred_df, actual_df], axis=1)


    daily_mse_list, daily_mape_list = [], []
    for i in range(len(predictions_list)):
        pred_single_day_array = predictions_list[i]
        act_single_day_array = actuals_list[i]

        daily_mse = mean_squared_error(act_single_day_array, pred_single_day_array)

        mask = act_single_day_array != 0
        if np.any(mask):
            mape_elements = np.abs((act_single_day_array[mask] - pred_single_day_array[mask]) / act_single_day_array[mask])
            mape = np.mean(mape_elements) * 100
        else:
            mape = 0.0 if np.all(pred_single_day_array == 0) else np.nan

        daily_mse_list.append(daily_mse)
        daily_mape_list.append(mape)

    errors_df = pd.DataFrame({'Daily_MSE': daily_mse_list, 'Daily_MAPE': daily_mape_list}, index=pd.DatetimeIndex(dates_list))

    print(f"Eval complete for {model_name}. Avg Overall Daily MSE: {errors_df['Daily_MSE'].mean():.4f}, Avg Overall Daily MAPE: {errors_df['Daily_MAPE'].mean():.2f}%")
    return errors_df, details_df

# chunk 5
def calculate_cohen_d(x, y):
    x = np.array(x).astype(float); y = np.array(y).astype(float)
    x = x[~np.isnan(x)]; y = y[~np.isnan(y)]
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2: return np.nan
    mean_x, mean_y = np.mean(x), np.mean(y)
    sd_x, sd_y = np.std(x, ddof=1), np.std(y, ddof=1)
    sd_x_is_zero = sd_x < 1e-9; sd_y_is_zero = sd_y < 1e-9
    if sd_x_is_zero and sd_y_is_zero: return 0.0 if np.isclose(mean_x, mean_y) else np.inf * np.sign(mean_x - mean_y)
    if (sd_x_is_zero and not np.isclose(mean_x, mean_y)) or \
       (sd_y_is_zero and not np.isclose(mean_x, mean_y)):
        return np.inf * np.sign(mean_x - mean_y)
    dof = nx + ny - 2
    if dof <= 0: return np.nan
    pooled_sd_numerator = 0
    if nx > 1: pooled_sd_numerator += (nx - 1) * sd_x**2
    if ny > 1: pooled_sd_numerator += (ny - 1) * sd_y**2
    if pooled_sd_numerator < 0 and pooled_sd_numerator > -1e-9: pooled_sd_numerator = 0.0
    if pooled_sd_numerator < 0: return np.nan
    pooled_sd = np.sqrt(pooled_sd_numerator / dof) if dof > 0 else 0
    if pooled_sd < 1e-9: return 0.0 if np.isclose(mean_x, mean_y) else np.inf * np.sign(mean_x - mean_y)
    return (mean_x - mean_y) / pooled_sd

def perform_statistical_analysis(results):
    """ Performs two-sample t-tests and Cohen's d on overall daily error metrics. """
    print("\n--- Statistical Analysis (QiLSTM vs StandardLSTM on Overall Daily Errors) ---")
    print("Lower MSE/MAPE is better. Negative t-stat means QiLSTM had lower mean error (better).")
    print("Cohen's d: small effect ~0.2, medium ~0.5, large ~0.8. Negative d means QiLSTM error < StandardLSTM error.")

    for period in ['Low Volatility', 'High Volatility']:
        print(f"\n{period} Period:")
        try:
            q_errors = results.get('QiLSTM', {}).get(period, {}).get('errors')
            s_errors = results.get('StandardLSTM', {}).get(period, {}).get('errors')

            if q_errors is None or q_errors.empty:
                print(f"  Skipping QiLSTM for {period} - Missing or empty evaluation results.")
                continue # Skip this period for QiLSTM, might still have StdLSTM
            if s_errors is None or s_errors.empty:
                print(f"  Skipping StandardLSTM for {period} - Missing or empty evaluation results.")
                continue # Skip this period for StdLSTM

            for metric in ['Daily_MSE', 'Daily_MAPE']:
                if metric not in q_errors.columns:
                    print(f"  {metric}: Skipped for QiLSTM (Metric column '{metric}' missing)")
                    continue_metric_q = False
                else:
                    q_metric_data = q_errors[metric].dropna()
                    continue_metric_q = True

                if metric not in s_errors.columns:
                    print(f"  {metric}: Skipped for StandardLSTM (Metric column '{metric}' missing)")
                    continue_metric_s = False
                else:
                    s_metric_data = s_errors[metric].dropna()
                    continue_metric_s = True

                if not (continue_metric_q and continue_metric_s):
                    continue


                n_q, n_s = len(q_metric_data), len(s_metric_data)
                if n_q < 2 or n_s < 2:
                    print(f"  {metric}: Skipped (Insufficient non-NaN data: QiLSTM n={n_q}, StdLSTM n={n_s})")
                    continue

                t_stat, p_value = stats.ttest_ind(q_metric_data, s_metric_data, equal_var=False, nan_policy='omit')
                d_value = calculate_cohen_d(q_metric_data, s_metric_data)
                sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
                direction = "QiLSTM better" if t_stat < 0 else "StdLSTM better" if t_stat > 0 else "No difference"
                print(f"  {metric}: t({n_q+n_s-2:.0f})={t_stat:.3f}, p={p_value:.3g}{sig}, Cohen's d={d_value:.3f}. {direction if sig else ''}")
        except KeyError as e:
            print(f"  Skipping {period} due to KeyError: {e} - Results dictionary incomplete.")
        except Exception as e:
            metric_name_for_error = metric if 'metric' in locals() else 'unknown metric'
            print(f"  Error during statistical analysis for {period}, metric {metric_name_for_error}: {e}")
            traceback.print_exc()

# chunk 6
def _configure_font_sizes(fig, base_size=12):
    """Applies consistent font sizing to a Plotly figure."""
    title_size = base_size + 6
    axis_title_size = base_size + 2
    subplot_title_size = base_size + 2
    tick_legend_size = base_size
    fig.update_layout(
        title_font_size=title_size,
        font_size=tick_legend_size,
        legend_font_size=tick_legend_size
    )
    fig.update_annotations(font_size=subplot_title_size)
    fig.update_xaxes(title_font_size=axis_title_size, tickfont_size=tick_legend_size)
    fig.update_yaxes(title_font_size=axis_title_size, tickfont_size=tick_legend_size)

def get_volatility_plot_components(vix_data, low_vol_range, high_vol_range, vix_ticker_name):
    """Returns traces, shapes, and layout updates for VIX plot."""
    print("Getting VIX plot components...")
    traces = []
    shapes = []
    layout_updates = {'xaxis_title': "Date", 'yaxis_title': "VIX Value"}

    if vix_ticker_name not in vix_data.columns:
        print(f"Error: VIX ticker '{vix_ticker_name}' not found in VIX data columns: {vix_data.columns}")
        return traces, shapes, layout_updates

    traces.append(go.Scatter(x=vix_data.index, y=vix_data[vix_ticker_name], mode='lines', name=f'{vix_ticker_name} Index'))

    shapes.append(dict(type="rect", x0=low_vol_range[0], x1=low_vol_range[1], y0=0, y1=1, yref="paper",
                       fillcolor="LightGreen", opacity=0.3, layer="below", line_width=0))
    shapes.append(dict(type="rect", x0=high_vol_range[0], x1=high_vol_range[1], y0=0, y1=1, yref="paper",
                       fillcolor="LightSalmon", opacity=0.3, layer="below", line_width=0))

    traces.append(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='LightGreen', size=10), name="Low Vol Period"))
    traces.append(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='LightSalmon', size=10), name="High Vol Period"))

    return traces, shapes, layout_updates

def get_loss_curves_plot_components(train_losses_q, val_losses_q, train_losses_s, val_losses_s):
    """Returns traces and layout updates for loss curves plot."""
    print("Getting loss curves plot components...")
    traces = []
    layout_updates = {'xaxis_title': 'Epoch', 'yaxis_title': 'MSE Loss (Log Scale)', 'yaxis_type': 'log'}

    has_q_data = bool(train_losses_q) and bool(val_losses_q)
    has_s_data = bool(train_losses_s) and bool(val_losses_s)

    if not has_q_data and not has_s_data:
        print("Warn: No loss data provided for either model for components.")
        return traces, layout_updates

    if has_q_data:
        epochs_q = list(range(1, len(train_losses_q) + 1))
        traces.append(go.Scatter(x=epochs_q, y=train_losses_q, mode='lines', name='QiLSTM Train Loss', line=dict(color='blue')))
        traces.append(go.Scatter(x=epochs_q, y=val_losses_q, mode='lines', name='QiLSTM Val Loss', line=dict(dash='dash', color='lightblue')))
    if has_s_data:
        epochs_s = list(range(1, len(train_losses_s) + 1))
        traces.append(go.Scatter(x=epochs_s, y=train_losses_s, mode='lines', name='Std LSTM Train Loss', line=dict(color='red')))
        traces.append(go.Scatter(x=epochs_s, y=val_losses_s, mode='lines', name='Std LSTM Val Loss', line=dict(dash='dash', color='salmon')))
    return traces, layout_updates

def get_prediction_comparison_plot_components(results, model1_name, model2_name, period_name, target_symbol):
    """Returns traces and layout updates for prediction comparison plot for a specific target_symbol."""
    print(f"Getting prediction comparison components for {target_symbol} ({period_name})...")
    traces = []
    layout_updates = {'yaxis_title': 'Price', 'xaxis_title': 'Date'}
    try:
        details1 = results.get(model1_name, {}).get(period_name, {}).get('details', pd.DataFrame())
        details2 = results.get(model2_name, {}).get(period_name, {}).get('details', pd.DataFrame())

        if details1.empty and details2.empty:
            print(f"Warn: Skipping plot components for {period_name} ({target_symbol}) - Details data missing.")
            return traces, layout_updates

        actual_col = f'{target_symbol}_Actual'
        pred1_col = f'{target_symbol}_Pred'
        pred2_col = f'{target_symbol}_Pred'

        actual_x, actual_y = None, None
        if not details1.empty and actual_col in details1.columns:
            actual_x, actual_y = details1.index, details1[actual_col]
        elif not details2.empty and actual_col in details2.columns:
            actual_x, actual_y = details2.index, details2[actual_col]

        if actual_x is not None:
            traces.append(go.Scatter(x=actual_x, y=actual_y, mode='lines', name=f'{target_symbol} Actual', line=dict(color='black')))
        else:
            print(f"ERR plot components: Actual column '{actual_col}' missing for {target_symbol} in {period_name}.")

        if not details1.empty and pred1_col in details1.columns:
            traces.append(go.Scatter(x=details1.index, y=details1[pred1_col], mode='lines', name=f'{model1_name} Pred ({target_symbol})', line=dict(dash='dot', color='blue')))

        if not details2.empty and pred2_col in details2.columns:
            traces.append(go.Scatter(x=details2.index, y=details2[pred2_col], mode='lines', name=f'{model2_name} Pred ({target_symbol})', line=dict(dash='dash', color='red')))
    except Exception as e:
        print(f"ERR plot components: Failed getting prediction components for {target_symbol} in {period_name}: {e}")
    return traces, layout_updates

def get_daily_errors_plot_components(results, model1_name, model2_name, period_name, metric):
    """
    Returns traces and layout updates for OVERALL daily errors plot.
    `metric` should be 'Daily_MSE' or 'Daily_MAPE'.
    """
    print(f"Getting overall daily error components for {metric} ({period_name})...")
    traces = []

    yaxis_title_text = f"Overall {metric}"
    yaxis_type_val = 'log' if metric == 'Daily_MSE' else 'linear'
    if metric == 'Daily_MSE': yaxis_title_text += " (Log Scale)"
    layout_updates = {'yaxis_title': yaxis_title_text, 'yaxis_type': yaxis_type_val, 'xaxis_title': 'Date'}

    try:
        errors1_df = results.get(model1_name, {}).get(period_name, {}).get('errors', pd.DataFrame())
        errors2_df = results.get(model2_name, {}).get(period_name, {}).get('errors', pd.DataFrame())

        if errors1_df.empty and errors2_df.empty:
            print(f"Warn: Skipping overall daily error components {period_name} ({metric}) - Error data missing for both models.")
            return traces, layout_updates

        if not errors1_df.empty and metric in errors1_df.columns:
            traces.append(go.Scatter(x=errors1_df.index, y=errors1_df[metric], mode='lines', name=f'{model1_name} ({metric})', line=dict(color='blue')))
        else: print(f"Warn: Metric '{metric}' missing for {model1_name} in {period_name} error data.")

        if not errors2_df.empty and metric in errors2_df.columns:
            traces.append(go.Scatter(x=errors2_df.index, y=errors2_df[metric], mode='lines', name=f'{model2_name} ({metric})', line=dict(color='red')))
        else: print(f"Warn: Metric '{metric}' missing for {model2_name} in {period_name} error data.")
    except Exception as e:
        print(f"ERR plot components: Failed getting overall daily error components for {period_name} ({metric}): {e}")
    return traces, layout_updates

# chunk 7
config = {
    'symbols': ['AAPL', 'MSFT', 'AMZN', 'NVDA', 'GOOGL', 'META', 'TSLA', 'SPY'],
    'vix_ticker': '^VIX',
    'start_date': datetime(2010, 1, 1),
    'data_end_date': datetime(2024, 3, 28),
    'training_val_split_date': datetime(2019, 12, 31),

    'low_vol_start': datetime(2023, 11, 1),
    'low_vol_end': datetime(2024, 3, 28),
    'high_vol_start': datetime(2020, 2, 19),
    'high_vol_end': datetime(2020, 4, 17),

    'seq_length': 60,
    'num_epochs_max': 200,
    'batch_size': 64,
    'patience_early_stopping': 40,

    'qilstm_params': {
        'hidden_size_lstm': 128,
        'num_layers_lstm': 2,
        'dropout_rate': 0.25,
        'quantum_complex_output_dim': 20,
        'theta_phi_init_scale': 0.03,
        'q_noise_std': 0.002,
        'learning_rate': 0.0006,
        'weight_decay': 2e-4,
    },
    'lstm_params': {
        'hidden_size': 128,
        'num_layers': 2,
        'dropout_rate': 0.2,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
    }
}

config['feature_cols'] = config['symbols'] + [config['vix_ticker']]
config['target_cols'] = config['symbols']
config['input_size'] = len(config['feature_cols'])
config['output_size'] = len(config['target_cols'])

# chunk 8
def main(config_dict):
    results = { model: {period: {'errors': pd.DataFrame(), 'details': pd.DataFrame()} for period in ['Low Volatility', 'High Volatility']} for model in ['QiLSTM', 'StandardLSTM']}
    training_history = {}
    try:
        print("--- Starting Analysis ---")
        full_data = fetch_data_and_vix(config_dict['symbols'], config_dict['vix_ticker'], config_dict['start_date'], config_dict['data_end_date'])
        vix_plot_data = full_data[[config_dict['vix_ticker']]].copy()
        feature_cols = config_dict['feature_cols']
        target_cols = config_dict['target_cols']
        input_size = config_dict['input_size']
        output_size = config_dict['output_size']

        train_loader, val_loader, feature_scaler, target_scaler = prepare_train_val_data(
            full_data, feature_cols, target_cols,
            config_dict['training_val_split_date'],
            config_dict['seq_length'],
            batch_size=config_dict['batch_size']
        )

        print("\nInitializing models...")
        q_params = config_dict['qilstm_params']
        model_qilstm = EnhancedQuantumInspiredLSTM(
            input_size=input_size, hidden_size_lstm=q_params['hidden_size_lstm'],
            num_layers_lstm=q_params['num_layers_lstm'], output_size_final=output_size,
            dropout_rate=q_params['dropout_rate'], quantum_complex_output_dim=q_params['quantum_complex_output_dim'],
            theta_phi_init_scale=q_params['theta_phi_init_scale'], q_noise_std=q_params['q_noise_std']
        ).to(device)
        l_params = config_dict['lstm_params']
        model_lstm = StandardLSTM(
            input_size=input_size, hidden_size=l_params['hidden_size'],
            num_layers=l_params['num_layers'], output_size=output_size,
            dropout_rate=l_params['dropout_rate']
        ).to(device)

        model_qilstm, train_loss_q, val_loss_q = train_model_with_val(
            model_qilstm, "QiLSTM", train_loader, val_loader,
            q_params['learning_rate'], q_params['weight_decay'],
            config_dict['num_epochs_max'], config_dict['patience_early_stopping']
        )
        training_history['QiLSTM'] = {'train': train_loss_q, 'val': val_loss_q}

        model_lstm, train_loss_s, val_loss_s = train_model_with_val(
            model_lstm, "StandardLSTM", train_loader, val_loader,
            l_params['learning_rate'], l_params['weight_decay'],
            config_dict['num_epochs_max'], config_dict['patience_early_stopping']
        )
        training_history['StandardLSTM'] = {'train': train_loss_s, 'val': val_loss_s}

        eval_periods = {
            'Low Volatility': (config_dict['low_vol_start'], config_dict['low_vol_end']),
            'High Volatility': (config_dict['high_vol_start'], config_dict['high_vol_end'])
        }
        for period_name, (start_date, end_date) in eval_periods.items():
            print(f"\n--- Evaluating for {period_name} ---")
            errors_q, details_q = evaluate_period(
                model_qilstm, f"QiLSTM Eval ({period_name.replace(' ', '')})", full_data,
                feature_cols, target_cols, start_date, end_date,
                config_dict['seq_length'], feature_scaler, target_scaler
            )
            results['QiLSTM'][period_name]['errors'] = errors_q
            results['QiLSTM'][period_name]['details'] = details_q

            errors_s, details_s = evaluate_period(
                model_lstm, f"StdLSTM Eval ({period_name.replace(' ', '')})", full_data,
                feature_cols, target_cols, start_date, end_date,
                config_dict['seq_length'], feature_scaler, target_scaler
            )
            results['StandardLSTM'][period_name]['errors'] = errors_s
            results['StandardLSTM'][period_name]['details'] = details_s

        print("\n--- Overall Performance Summary (Based on Overall Daily Price Errors) ---")
        summary_data = []
        for model_name_key in ['QiLSTM', 'StandardLSTM']:
            for period_name_key in ['Low Volatility', 'High Volatility']:
                err_df = results[model_name_key][period_name_key]['errors']
                avg_mse = err_df['Daily_MSE'].mean() if not err_df.empty and 'Daily_MSE' in err_df else np.nan
                avg_mape = err_df['Daily_MAPE'].mean() if not err_df.empty and 'Daily_MAPE' in err_df else np.nan
                summary_data.append({
                    'Model': model_name_key, 'Period': period_name_key,
                    'Avg Overall MSE': f"{avg_mse:.4f}" if not np.isnan(avg_mse) else "N/A",
                    'Avg Overall MAPE (%)': f"{avg_mape:.2f}" if not np.isnan(avg_mape) else "N/A"
                })
        summary_df = pd.DataFrame(summary_data)
        if not summary_df.empty:
            print(summary_df.to_string(index=False))
        else:
            print("No overall summary data to display.")

        perform_statistical_analysis(results) # Now uses overall daily errors

        # --- Generating Combined Visualizations ---
        print("\n--- Generating Combined Visualizations ---")
        plot_target_symbol = 'SPY' if 'SPY' in config_dict['target_cols'] else config_dict['target_cols'][0]

        # --- Master Figure 1: VIX & Losses ---
        print("Generating Figure 1: VIX and Model Loss Curves...")
        vix_traces, vix_shapes, vix_layout_updates = get_volatility_plot_components(
            vix_plot_data, (config_dict['low_vol_start'], config_dict['low_vol_end']),
            (config_dict['high_vol_start'], config_dict['high_vol_end']), config_dict['vix_ticker'])

        loss_traces, loss_layout_updates = get_loss_curves_plot_components(
            training_history.get('QiLSTM',{}).get('train',[]), training_history.get('QiLSTM',{}).get('val',[]),
            training_history.get('StandardLSTM',{}).get('train',[]), training_history.get('StandardLSTM',{}).get('val',[]))

        if vix_traces or loss_traces:
            fig1 = make_subplots(rows=2, cols=1,
                                 subplot_titles=[f"A) {config_dict['vix_ticker']} Volatility Index",
                                                 "B) Model Training & Validation Loss (MSE)"],
                                 vertical_spacing=0.12, row_heights=[0.4, 0.6])
            if vix_traces:
                for trace in vix_traces: fig1.add_trace(trace, row=1, col=1)
                for shape in vix_shapes: fig1.add_shape(shape, row=1, col=1)
                fig1.update_xaxes(title_text=vix_layout_updates.get('xaxis_title',"Date"), row=1, col=1)
                fig1.update_yaxes(title_text=vix_layout_updates.get('yaxis_title',"VIX Value"), row=1, col=1)
            if loss_traces:
                for trace in loss_traces: fig1.add_trace(trace, row=2, col=1)
                fig1.update_xaxes(title_text=loss_layout_updates.get('xaxis_title',"Epoch"), row=2, col=1)
                fig1.update_yaxes(title_text=loss_layout_updates.get('yaxis_title',"MSE Loss (Log Scale)"),
                                  type=loss_layout_updates.get('yaxis_type','log'), row=2, col=1)

            fig1.update_layout(title_text="Figure 1: Market Volatility and Model Training Performance", height=800, showlegend=True)
            _configure_font_sizes(fig1)
            fig1.show()
        else:
            print("Skipping Figure 1: No VIX or Loss data to plot.")

        # --- Master Figure 2 & 3: Low/High Volatility Analysis ---
        # Panel A: Prediction for plot_target_symbol
        # Panel B & C: Overall Daily MSE and MAPE
        for period_key, period_label, fig_num in [
            ('Low Volatility', 'Low Volatility Period', 2),
            ('High Volatility', 'High Volatility Period', 3)
        ]:
            print(f"Generating Figure {fig_num}: {period_label} Analysis...")
            # Panel A: Specific symbol prediction
            pred_traces, pred_layout = get_prediction_comparison_plot_components(results, 'QiLSTM', 'StandardLSTM', period_key, plot_target_symbol)
            # Panel B: Overall Daily MSE
            mse_traces, mse_layout = get_daily_errors_plot_components(results, 'QiLSTM', 'StandardLSTM', period_key, 'Daily_MSE')
            # Panel C: Overall Daily MAPE
            mape_traces, mape_layout = get_daily_errors_plot_components(results, 'QiLSTM', 'StandardLSTM', period_key, 'Daily_MAPE')

            if not (pred_traces or mse_traces or mape_traces):
                print(f"Skipping Figure {fig_num}: No data to plot for {period_label}.")
                continue

            fig_combined = make_subplots(rows=3, cols=1,
                                         subplot_titles=[f"A) Price Prediction vs Actual ({plot_target_symbol})",
                                                         f"B) Overall Daily MSE",
                                                         f"C) Overall Daily MAPE (%)"],
                                         vertical_spacing=0.1, shared_xaxes=True)
            if pred_traces:
                for trace in pred_traces: fig_combined.add_trace(trace, row=1, col=1)
                fig_combined.update_yaxes(title_text=pred_layout.get('yaxis_title', 'Price'), row=1, col=1)
            if mse_traces:
                for trace in mse_traces: fig_combined.add_trace(trace, row=2, col=1)
                fig_combined.update_yaxes(title_text=mse_layout.get('yaxis_title', 'Overall Daily MSE'), type=mse_layout.get('yaxis_type', 'log'), row=2, col=1)
            if mape_traces:
                for trace in mape_traces: fig_combined.add_trace(trace, row=3, col=1)
                fig_combined.update_yaxes(title_text=mape_layout.get('yaxis_title', 'Overall Daily MAPE (%)'), type=mape_layout.get('yaxis_type', 'linear'), row=3, col=1)

            fig_combined.update_xaxes(title_text="Date", row=3, col=1)
            fig_combined.update_layout(title_text=f"Figure {fig_num}: Performance Analysis during {period_label}", height=1000, showlegend=True)
            _configure_font_sizes(fig_combined)
            fig_combined.show()

        print("\n--- Analysis Complete ---")

    except Exception as e:
        print(f"\n--- An Error Occurred in Main Execution ---")
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {e}")
        print("\nTraceback:")
        print(traceback.format_exc())
        print("--------------------------")

if __name__ == "__main__":
    main(config)

[***********           22%                       ]  2 of 9 completed

Using device: cuda
--- Starting Analysis ---
--- Fetching Data ---
Assets: AAPL, MSFT, AMZN, NVDA, GOOGL, META, TSLA, SPY, ^VIX
Period: 2010-01-01 to 2024-03-28


[*********************100%***********************]  9 of 9 completed


Dropped 599 rows due to missing values after combining.
--- Data Fetching Complete --- Final Shape: (2983, 9)
--- Preparing Train/Val Data (Split Date: 2019-12-31) ---
DataLoaders created.

Initializing models...

--- Starting Training for QiLSTM ---
QiLSTM - Epoch [1/200], Train Loss: 0.119203, Val Loss: 0.016051, LR: 0.0006
QiLSTM - Epoch [10/200], Train Loss: 0.007097, Val Loss: 0.002776, LR: 0.0006
QiLSTM - Epoch [20/200], Train Loss: 0.005180, Val Loss: 0.002294, LR: 0.0006
QiLSTM - Epoch [30/200], Train Loss: 0.004011, Val Loss: 0.001499, LR: 0.0006
QiLSTM - Epoch [40/200], Train Loss: 0.003676, Val Loss: 0.001179, LR: 0.0006
QiLSTM - Epoch [50/200], Train Loss: 0.002960, Val Loss: 0.000864, LR: 0.0006
QiLSTM - Epoch [60/200], Train Loss: 0.002872, Val Loss: 0.000756, LR: 0.0006
QiLSTM - Epoch [70/200], Train Loss: 0.002442, Val Loss: 0.000535, LR: 0.0003
QiLSTM - Epoch [80/200], Train Loss: 0.002404, Val Loss: 0.000675, LR: 0.0003
QiLSTM - Epoch [90/200], Train Loss: 0.002433, V

Generating Figure 2: Low Volatility Period Analysis...
Getting prediction comparison components for SPY (Low Volatility)...
Getting overall daily error components for Daily_MSE (Low Volatility)...
Getting overall daily error components for Daily_MAPE (Low Volatility)...


Generating Figure 3: High Volatility Period Analysis...
Getting prediction comparison components for SPY (High Volatility)...
Getting overall daily error components for Daily_MSE (High Volatility)...
Getting overall daily error components for Daily_MAPE (High Volatility)...



--- Analysis Complete ---
